# Hydroclimatic memory in stream chemistry — reproducibility notebook

This notebook accompanies the manuscript **“Hydroclimatic memory in stream chemistry persists beyond discharge and varies with catchment organization.”**

It is the portable public entry point to the frozen analysis package. It:

1. loads the frozen analysis-ready inputs using repository-relative paths;
2. verifies the monitoring architecture and the pre-specified response panel;
3. checks the 29 antecedent hydroclimate windows represented in the analysis dataset;
4. exposes censoring, discharge and station-network inputs;
5. loads the frozen inferential and LOCO products used in the manuscript;
6. reproduces the reported strict-memory, discharge-conditioning and contributing-area result tables and displays the final figures.

The historical development notebook and raw laboratory workbooks are intentionally not required for this public workflow. The CSV files in `data/processed/` are the frozen analysis inputs; `results/` contains the frozen inferential outputs used for the manuscript.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    raise FileNotFoundError(
        "Run this notebook from the repository root so data/processed is available."
    )

DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"

print("Repository root:", ROOT.resolve())
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 1. Load frozen analysis inputs


In [ ]:
files = {
    "memory": DATA / "MEMORY_ANALYSIS_DATASET_v1.csv",
    "daily": DATA / "HYDROCLIMATE_DAILY_MASTER_v1.csv",
    "windows": DATA / "HYDROCLIMATE_EVENT_ANTECEDENT_WINDOWS_v1.csv",
    "hydrometry": DATA / "HYDROMETRY_FROZEN_v1.csv",
    "stations": DATA / "STATION_NETWORK_FROZEN_v1.csv",
    "panel": DATA / "PRIMARY_MEMORY_RESPONSE_PANEL_v1.csv",
    "censoring": DATA / "CENSORING_POLICY_FINAL_v1.csv",
    "dictionary": DATA / "PARAMETER_DICTIONARY_FINAL_v2.csv",
}
missing = [str(p) for p in files.values() if not p.exists()]
assert not missing, f"Missing frozen inputs: {missing}"

D = {k: pd.read_csv(v, low_memory=False) for k, v in files.items()}
pd.DataFrame({
    "object": D.keys(),
    "rows": [len(x) for x in D.values()],
    "columns": [x.shape[1] for x in D.values()],
})


## 2. Frozen monitoring architecture


In [ ]:
memory = D["memory"]
architecture = pd.Series({
    "analysis_rows": len(memory),
    "unique_physical_stations": memory["physical_station_id_v1"].nunique(),
    "campaigns": memory["campaign_id"].nunique(),
    "responses": memory["response_key"].nunique(),
    "hydroclimate_event_dates": memory["hydroclimate_event_date"].nunique(),
})
architecture


The public analysis dataset is in long response format: each eligible field/sample identity can contribute one row per response variable. Counts therefore should not be interpreted as numbers of independent sampling events without resolving the corresponding identity columns.


## 3. Pre-specified response panel


In [ ]:
panel = D["panel"].copy()
panel[["response_key", "response_family", "memory_panel_role", "scientific_role",
       "freeze_status", "automatic_selection"]]


## 4. Antecedent hydroclimate windows


In [ ]:
window_cols = [
    c for c in D["windows"].columns
    if re.match(r"^(ppt_sum|sm_surface_mean|sm_rootzone_mean|tair_mean|pet_mean)_\d+d$", c)
]
# Fall back to the frozen memory dataset naming if needed.
if not window_cols:
    window_cols = [
        c for c in memory.columns
        if re.search(r"(ppt|precip|sm_|soil|tair|temp|pet).*(1d|3d|7d|14d|30d|60d|90d)$", c, re.I)
    ]
print("Detected antecedent-window columns:", len(window_cols))
print(window_cols)


## 5. Hydrometry and station-network inputs


In [ ]:
hydrometry = D["hydrometry"]
stations = D["stations"]

q_summary = pd.Series({
    "hydrometry_rows": len(hydrometry),
    "observed_Q": hydrometry["discharge_observed"].notna().sum(),
    "reconstructed_Q": hydrometry["q_is_reconstructed"].fillna(False).astype(bool).sum(),
    "physical_stations": len(stations),
})
q_summary


## 6. Censoring policy


In [ ]:
censor = D["censoring"]
censor.loc[censor["parameter_std"].isin(panel["response_key"]) |
           censor["parameter_id"].isin(panel["response_key"]),
           ["parameter_id","parameter_std","n_records","n_censored","pct_censored",
            "censoring_class","censor_policy_final"]].head(30)


## 7. Frozen inferential products

The computationally intensive Bayesian fitting and leave-one-campaign-out (LOCO) validation were frozen before manuscript closure. The following files are the exact manuscript-facing outputs retained from that analysis. Keeping these products alongside the frozen inputs makes the numerical claims auditable without requiring hundreds of model refits merely to inspect the published result.


In [ ]:
def load_result(folder, name):
    p = RESULTS / folder / name
    if not p.exists():
        raise FileNotFoundError(p)
    return pd.read_csv(p, low_memory=False)

strict_memory = load_result("main", "TABLE_MEMORY_STRICT_HYDROCHEMICAL_FINAL.csv")
hq = load_result("main", "TABLE_02_ANTECEDENT_HYDROCLIMATE_BEYOND_Q_FINAL.csv")
cq = load_result("main", "TABLE_CQ_DISCHARGE_RESPONSE_SPECTRUM_FINAL.csv")
h3 = load_result("main", "TABLE_H3_CONTRIBUTING_AREA_MODULATION_FINAL.csv")

print("Strict hydrochemical memory")
display(strict_memory)
print("\nAntecedent hydroclimate beyond contemporaneous Q")
display(hq)
print("\nConcentration–discharge response spectrum")
display(cq)
print("\nContributing-area modulation")
display(h3)


## 8. LOCO support and computational QA


In [ ]:
loco_hq = load_result("supplementary", "TABLE_S04_BLOCK10_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv")
qa_hq = load_result("supplementary", "TABLE_S05_BLOCK10_COMPUTATIONAL_QA.csv")
loco_h3 = load_result("supplementary", "TABLE_S06_H3_ALL_50_CAMPAIGN_LOCO_CONTRASTS.csv")
qa_h3 = load_result("supplementary", "TABLE_S07_H3_COMPUTATIONAL_QA_100_FITS.csv")

pd.DataFrame({
    "product": ["H+Q campaign LOCO", "H+Q computational QA",
                "H×area campaign LOCO", "H×area computational QA"],
    "rows": [len(loco_hq), len(qa_hq), len(loco_h3), len(qa_h3)]
})


LOCO means **leave one complete sampling campaign out**. Whole campaigns, rather than individual observations, were withheld so that observations sharing the same event-scale hydroclimatic forcing could not leak between training and validation sets.


## 9. Final manuscript figures


In [ ]:
figure_files = [
    FIGURES / "FIGURE_02_HYDROCLIMATIC_MEMORY_SPECTRUM_FINAL.png",
    FIGURES / "FIGURE_03_CQ_RESPONSE_SPECTRUM_EDITORIAL_FINAL.png",
    FIGURES / "FIGURE_04_CONNECTIVITY_MODULATION_EDITORIAL_FINAL.png",
]
for p in figure_files:
    if p.exists():
        img = Image.open(p)
        plt.figure(figsize=(12, 7))
        plt.imshow(img)
        plt.axis("off")
        plt.title(p.stem)
        plt.show()


## 10. Interpretation boundaries

- The selected antecedent window is treated as a **predictive hydroclimatic-memory scale**, not as a direct estimate of water residence time.
- Antecedent hydroclimate is shared event-date forcing over the Razuhuillca reference domain; it is not a station-specific upstream forcing reconstruction.
- Contributing area is used as a structural catchment proxy. Its interaction with antecedent hydroclimate is compatible with changes in storage, mixing or flow-path integration, but does not by itself identify a causal connectivity mechanism.
- Primary discharge-conditioned inference uses observed discharge. Reconstructed discharge is reserved for sensitivity analysis where specified.


## Citation and reuse

Please cite the associated manuscript and the archived software release/DOI when available. The repository is released under the MIT License; data reuse should also preserve the scientific definitions and interpretation boundaries documented here.
